In [1]:
import requests

/Users/blanca/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Importing keys and tokens 

In [5]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

my_key_tv_maze = os.getenv("API_KEY_TV_MAZE")
my_token_tmbd = os.getenv("API_TOKEN_TMDB")
my_key_tmdb = os.getenv("API_KEY_TMDB")

create the CSV of data 

TMBD

In [ ]:
# extract_tmdb.py
from __future__ import annotations

import os
import time
import requests
import pandas as pd
from typing import Any, Dict, List, Optional


# =========================
# CONFIG
# =========================
my_token_tmbd = os.getenv("API_TOKEN_TMDB")
BASE_URL = "https://api.themoviedb.org/3"

# How many pages to collect from discover endpoints
MOVIE_PAGES = 500
TV_PAGES = 383

# Sleep a bit between requests to be polite
REQUEST_SLEEP = 0.20


# =========================
# HTTP HELPERS
# =========================
def tmdb_headers() -> Dict[str, str]:
    return {
        "accept": "application/json",
        "Authorization": f"Bearer {my_token_tmbd}",
    }


def tmdb_get(endpoint: str, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    url = f"{BASE_URL}{endpoint}"
    response = requests.get(url, headers=tmdb_headers(), params=params, timeout=30)
    response.raise_for_status()
    time.sleep(REQUEST_SLEEP)
    return response.json()


# =========================
# LOOKUPS
# =========================
def get_genre_map() -> Dict[int, str]:
    genre_map: Dict[int, str] = {}

    movie_genres = tmdb_get("/genre/movie/list").get("genres", [])
    tv_genres = tmdb_get("/genre/tv/list").get("genres", [])

    for g in movie_genres + tv_genres:
        genre_map[g["id"]] = g["name"]

    return genre_map


# =========================
# EXTRACTION
# =========================
def fetch_discover_titles(content_type: str, pages: int) -> List[Dict[str, Any]]:
    """
    content_type: 'movie' or 'tv'
    """
    all_results: List[Dict[str, Any]] = []

    endpoint = f"/discover/{content_type}"

    for page in range(1, pages + 1):
        data = tmdb_get(
            endpoint,
            params={
                "language": "en-US",
                "sort_by": "popularity.desc",
                "page": page,
                "include_adult": "false",
            },
        )
        results = data.get("results", [])
        all_results.extend(results)
        print(f"[TMDb] fetched {content_type} discover page {page} -> {len(results)} rows")

    return all_results


def fetch_details(content_type: str, item_id: int) -> Dict[str, Any]:
    endpoint = f"/{content_type}/{item_id}"
    return tmdb_get(endpoint, params={"language": "en-US"})


def safe_join(values: Optional[List[str]]) -> Optional[str]:
    if not values:
        return None
    return ", ".join(v for v in values if v)


def transform_tmdb_item(
    base_item: Dict[str, Any],
    details: Dict[str, Any],
    genre_map: Dict[int, str],
    content_type: str,
) -> Dict[str, Any]:
    # Unified title and date fields
    title = base_item.get("title") or base_item.get("name")
    release_date = base_item.get("release_date") or base_item.get("first_air_date")

    # Genre IDs from base result
    genre_ids = base_item.get("genre_ids", [])
    genre_names = [genre_map.get(gid, f"unknown_{gid}") for gid in genre_ids]

    # Detail-level genres if available
    detail_genres = details.get("genres", [])
    detail_genre_names = [g.get("name") for g in detail_genres if g.get("name")]

    # origin country exists more often on TV details
    origin_countries = details.get("origin_country", [])
    if not origin_countries and base_item.get("origin_country"):
        origin_countries = base_item.get("origin_country", [])

    row = {
        "source": "tmdb",
        "id": base_item.get("id"),
        "title": title,
        "original_title": details.get("original_title") or details.get("original_name"),
        "content_type": "movie" if content_type == "movie" else "tv",
        "genre_ids": genre_ids,
        "genre_names": safe_join(detail_genre_names or genre_names),
        "original_language": base_item.get("original_language"),
        "popularity": base_item.get("popularity"),
        "vote_average": base_item.get("vote_average"),
        "vote_count": base_item.get("vote_count"),
        "release_date": release_date,
        "overview": base_item.get("overview"),
        "adult": base_item.get("adult"),
        "origin_country": safe_join(origin_countries),
        "tmdb_id": base_item.get("id"),
        # extra useful enrichment
        "status": details.get("status"),
        "runtime": details.get("runtime"),  # movie
        "number_of_seasons": details.get("number_of_seasons"),  # tv
        "number_of_episodes": details.get("number_of_episodes"),  # tv
        "original_name": details.get("original_name"),
        "homepage": details.get("homepage"),
        "imdb_id": details.get("imdb_id"),
        "production_companies": safe_join(
            [c.get("name") for c in details.get("production_companies", []) if c.get("name")]
        ),
        "production_countries": safe_join(
            [c.get("iso_3166_1") for c in details.get("production_countries", []) if c.get("iso_3166_1")]
        ),
        "spoken_languages": safe_join(
            [c.get("english_name") for c in details.get("spoken_languages", []) if c.get("english_name")]
        ),
    }

    return row


def build_tmdb_dataset(movie_pages: int = 10, tv_pages: int = 10) -> pd.DataFrame:
    genre_map = get_genre_map()

    movie_items = fetch_discover_titles("movie", movie_pages)
    tv_items = fetch_discover_titles("tv", tv_pages)

    all_rows: List[Dict[str, Any]] = []

    for item in movie_items:
        try:
            details = fetch_details("movie", item["id"])
            row = transform_tmdb_item(item, details, genre_map, "movie")
            all_rows.append(row)
        except Exception as e:
            print(f"[TMDb][movie] failed id={item.get('id')}: {e}")

    for item in tv_items:
        try:
            details = fetch_details("tv", item["id"])
            row = transform_tmdb_item(item, details, genre_map, "tv")
            all_rows.append(row)
        except Exception as e:
            print(f"[TMDb][tv] failed id={item.get('id')}: {e}")

    df = pd.DataFrame(all_rows)

    # Cleaning / standardization
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
    df["release_year"] = df["release_date"].dt.year

    df = df.drop_duplicates(subset=["source", "id", "content_type"]).copy()
    df = df[df["title"].notna() & (df["title"].str.strip() != "")].copy()

    return df


if __name__ == "__main__": # SABINA COMMENT: This specific way of writing code is more used for Python scripts than Jupyter notebooks
                            # In a Jupyter notebook it doesn't really make sense to have the if __name__ == "__main__", which basically means that if I 
                            # run "python name_of_script.py" in the terminal it's only going to run the lines of code afterwards
    if "my_token_tmbd" in my_token_tmbd:
        raise ValueError("Please set my_token_tmbd in your environment or inside the script.")

    df_tmdb = build_tmdb_dataset(movie_pages=MOVIE_PAGES, tv_pages=TV_PAGES)
    output_file = "tmdb_titles.csv"
    df_tmdb.to_csv(output_file, index=False, encoding="utf-8-sig")

    print(f"\nSaved {len(df_tmdb):,} rows to {output_file}")
    print(df_tmdb.head())

[TMDb] fetched movie discover page 1 -> 20 rows
[TMDb] fetched movie discover page 2 -> 20 rows
[TMDb] fetched movie discover page 3 -> 20 rows
[TMDb] fetched movie discover page 4 -> 20 rows
[TMDb] fetched movie discover page 5 -> 20 rows
[TMDb] fetched movie discover page 6 -> 20 rows
[TMDb] fetched movie discover page 7 -> 20 rows
[TMDb] fetched movie discover page 8 -> 20 rows
[TMDb] fetched movie discover page 9 -> 20 rows
[TMDb] fetched movie discover page 10 -> 20 rows
[TMDb] fetched movie discover page 11 -> 20 rows
[TMDb] fetched movie discover page 12 -> 20 rows
[TMDb] fetched movie discover page 13 -> 20 rows
[TMDb] fetched movie discover page 14 -> 20 rows
[TMDb] fetched movie discover page 15 -> 20 rows
[TMDb] fetched movie discover page 16 -> 20 rows
[TMDb] fetched movie discover page 17 -> 20 rows
[TMDb] fetched movie discover page 18 -> 20 rows
[TMDb] fetched movie discover page 19 -> 20 rows
[TMDb] fetched movie discover page 20 -> 20 rows
[TMDb] fetched movie discover

TV maze

In [13]:
import requests
import pandas as pd
import os
import time
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Get API key
my_key_tv_maze = os.getenv("API_KEY_TV_MAZE")

BASE_URL = "https://api.tvmaze.com"


# -----------------------------
# Helper Functions
# -----------------------------
def safe_get(dct, *keys):
    """Safely access nested dictionary keys."""
    for key in keys:
        if not isinstance(dct, dict):
            return None
        dct = dct.get(key)
    return dct


def clean_html(text):
    """Remove HTML tags from summaries."""
    if not isinstance(text, str):
        return text

    import re
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# -----------------------------
# Fetch shows
# -----------------------------
def fetch_shows(start_page=0, end_page=363):

    all_rows = []

    headers = {
        "Authorization": f"Bearer {my_key_tv_maze}"
    }

    for page in range(start_page, end_page + 1):

        url = f"{BASE_URL}/shows?page={page}"

        response = requests.get(url, headers=headers)

        if response.status_code != 200:
            print(f"Error on page {page}")
            break

        data = response.json()

        print(f"Fetched page {page} -> {len(data)} shows")

        for show in data:

            row = {
                "source": "tvmaze",
                "id": show.get("id"),
                "title": show.get("name"),
                "content_type": "tv",

                "genre_names": ", ".join(show.get("genres", [])),
                "original_language": show.get("language"),

                "popularity": None,   # not available in TVMaze
                "vote_average": safe_get(show, "rating", "average"),
                "vote_count": None,

                "release_date": show.get("premiered"),
                "release_year": None,

                "overview": clean_html(show.get("summary")),
                "origin_country": safe_get(show, "network", "country", "code"),

                # extra useful features
                "status": show.get("status"),
                "runtime": show.get("runtime"),
                "average_runtime": show.get("averageRuntime"),
                "network": safe_get(show, "network", "name"),
                "web_channel": safe_get(show, "webChannel", "name"),

                "official_site": show.get("officialSite"),
                "imdb_id": safe_get(show, "externals", "imdb"),
                "tvdb_id": safe_get(show, "externals", "thetvdb")
            }

            all_rows.append(row)

        time.sleep(0.25)

    df = pd.DataFrame(all_rows)

    # Convert dates
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
    df["release_year"] = df["release_date"].dt.year

    # Remove duplicates
    df = df.drop_duplicates(subset=["id"])

    return df


# -----------------------------
# Run extraction
# -----------------------------
if __name__ == "__main__":

    df = fetch_shows(0, 363)

    df.to_csv("tvmaze_titles.csv", index=False)

    print("Dataset saved: tvmaze_titles.csv")
    print(df.head())

Fetched page 0 -> 240 shows
Fetched page 1 -> 245 shows
Fetched page 2 -> 242 shows
Fetched page 3 -> 243 shows
Fetched page 4 -> 238 shows
Fetched page 5 -> 233 shows
Fetched page 6 -> 234 shows
Fetched page 7 -> 241 shows
Fetched page 8 -> 232 shows
Fetched page 9 -> 231 shows
Fetched page 10 -> 229 shows
Fetched page 11 -> 228 shows
Fetched page 12 -> 226 shows
Fetched page 13 -> 235 shows
Fetched page 14 -> 228 shows
Fetched page 15 -> 229 shows
Fetched page 16 -> 232 shows
Fetched page 17 -> 242 shows
Fetched page 18 -> 241 shows
Fetched page 19 -> 235 shows
Fetched page 20 -> 241 shows
Fetched page 21 -> 241 shows
Fetched page 22 -> 235 shows
Fetched page 23 -> 243 shows
Fetched page 24 -> 236 shows
Fetched page 25 -> 228 shows
Fetched page 26 -> 232 shows
Fetched page 27 -> 229 shows
Fetched page 28 -> 231 shows
Fetched page 29 -> 238 shows
Fetched page 30 -> 237 shows
Fetched page 31 -> 234 shows
Fetched page 32 -> 229 shows
Fetched page 33 -> 238 shows
Fetched page 34 -> 239 s